# Early Learning Support: Rebuild the Data

[Open in Colab](https://colab.research.google.com/github/Junghwan0516/early-learning-support/blob/main/00_data_preparation.ipynb)

This notebook reconstructs the prepared feature table from the official OULAD archive. The compressed download is approximately 45 MB; the large activity CSV is processed in chunks. CPU is sufficient.

The observation window is days 0–27. Learners must register by day 0 and remain enrolled at the start of day 28. Known withdrawals on or after day 28 are the positive target. Fail/Pass/Distinction are negative. Missing withdrawal dates and inconsistent outcome records are excluded.

Do not upload the raw archive to GitHub. Source: Kuzilek, Hlosta & Zdrahal (2015), https://doi.org/10.24432/C5KK69, CC BY 4.0.

In [1]:
from pathlib import Path
import urllib.request
import sys

repository_root = 'https://raw.githubusercontent.com/Junghwan0516/early-learning-support/main/'
for filename in ['els.py', 'prepare_data.py']:
    if not Path(filename).exists():
        with urllib.request.urlopen(repository_root + filename, timeout=60) as response:
            Path(filename).write_bytes(response.read())
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import els
els.style()

In [2]:
import prepare_data
import hashlib

archive = prepare_data.download('data/raw/oulad.zip')
print('Verified source archive:', archive)
rebuilt = prepare_data.make_features(archive)
supplied = els.load_data()
pd.testing.assert_frame_equal(rebuilt, supplied, check_dtype=False)
print('All rebuilt feature values and row order match the supplied CSV.')
print('Prepared records:', len(rebuilt))

Verified source archive: data/raw/oulad.zip
All rebuilt feature values and row order match the supplied CSV.
Prepared records: 5062


In [3]:
parts, split_audit = prepare_data.temporal_splits(rebuilt)
display(split_audit)
for earlier, later in [('train','validation'),('train','test'),('validation','test')]:
    assert set(parts[earlier].id_student).isdisjoint(parts[later].id_student)
print('No learner overlap between splits.')
split_audit.to_csv('reports/split_audit.csv', index=False)

No learner overlap between splits.


,split,presentation,eligible_before_dedup,excluded_repeat_learners,rows,withdrawn,withdrawal_rate
0,train,2013B,1522,0,1522,269,0.176741
1,validation,2013J,1816,28,1788,233,0.130313
2,test,2014J,1724,39,1685,219,0.129970


## Interpretation

The prepared table contains 5,062 learner-presentation records. It contains repeated learners across presentations, so it is not a count of distinct people. Removing earlier learners from later presentations leaves 1,522 training, 1,788 validation, and 1,685 final-test records.

The split audit checks cohort membership, not predictive performance. Merely sorting presentations does not establish when all training labels would have become available in real deployment.